In [0]:
# ============================================
# FORGEAXIS - BRONZE TO SILVER
# ============================================

from pyspark.sql.functions import (
    col,
    trim,
    lower,
    to_date,
    to_timestamp
)

# Create Silver schema
spark.sql("""
CREATE SCHEMA IF NOT EXISTS forgeaxis.forgeaxis_silver
""")


# ============================================
# 1. CUSTOMERS
# ============================================

customers_bronze_df = spark.table(
    "forgeaxis.forgeaxis_bronze.bronze_customer"
)

customers_silver_df = (
    customers_bronze_df
    .dropDuplicates(["customer_id"])
    .filter(col("customer_id").isNotNull())
    .filter(col("customer_name").isNotNull())
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("city", trim(col("city")))
    .withColumn("customer_type", trim(col("customer_type")))
    .withColumn("status", lower(trim(col("status"))))
    .withColumn("credit_limit", col("credit_limit").cast("double"))
    .filter(col("credit_limit").isNull() | (col("credit_limit") >= 0))
)

customers_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "forgeaxis.forgeaxis_silver.silver_customer"
    )


# ============================================
# 2. PRODUCTS
# ============================================

products_bronze_df = spark.table(
    "forgeaxis.forgeaxis_bronze.bronze_products"
)

products_silver_df = (
    products_bronze_df
    .dropDuplicates(["product_id"])
    .filter(col("product_id").isNotNull())
    .filter(col("product_name").isNotNull())
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("categories", trim(col("categories")))
    .withColumn("unit_price", col("unit_price").cast("double"))
    .filter(
        col("unit_price").isNotNull() &
        (col("unit_price") > 0)
    )
)

products_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "forgeaxis.forgeaxis_silver.silver_products"
    )


# ============================================
# 3. MACHINES
# ============================================

machines_bronze_df = spark.table(
    "forgeaxis.forgeaxis_bronze.bronze_machine"
)

machines_silver_df = (
    machines_bronze_df
    .dropDuplicates(["machine_id"])
    .filter(col("machine_id").isNotNull())
    .withColumn("machine_name", trim(col("machine_name")))
    .withColumn("production_line", trim(col("production_line")))
    .withColumn(
        "capacity_per_hour",
        col("capacity_per_hour").cast("int")
    )
    .filter(col("capacity_per_hour") > 0)
)

machines_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "forgeaxis.forgeaxis_silver.silver_machine"
    )


# ============================================
# 4. SALES ORDERS
# ============================================

sales_bronze_df = spark.table(
    "forgeaxis.forgeaxis_bronze.bronze_sales_orders"
)

sales_silver_df = (
    sales_bronze_df
    .dropDuplicates(["sales_order_id"])
    .filter(col("sales_order_id").isNotNull())
    .withColumn(
        "quantity_ordered",
        col("quantity_ordered").cast("int")
    )
    .withColumn(
        "unit_price",
        col("unit_price").cast("double")
    )
    .withColumn(
        "total_amount",
        col("total_amount").cast("double")
    )
    .withColumn(
        "order_date",
        to_date(col("order_date"), "dd-MM-yyyy")
    )
    .withColumn(
        "required_date",
        to_date(col("required_date"), "dd-MM-yyyy")
    )
    .withColumn(
        "order_status",
        lower(trim(col("order_status")))
    )
    .filter(
        (col("quantity_ordered") > 0) &
        (col("unit_price") > 0) &
        (col("total_amount") >= 0) &
        col("order_date").isNotNull() &
        col("required_date").isNotNull() &
        (col("required_date") >= col("order_date"))
    )
)

sales_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "forgeaxis.forgeaxis_silver.silver_sales_orders"
    )


# ============================================
# 5. PRODUCTION ORDERS
# ============================================

production_bronze_df = spark.table(
    "forgeaxis.forgeaxis_bronze.bronze_production_orders"
)

production_silver_df = (
    production_bronze_df
    .dropDuplicates(["production_order_id"])
    .filter(col("production_order_id").isNotNull())
    .filter(col("sales_order_id").isNotNull())
    .filter(col("machine_id").isNotNull())
    .withColumn(
        "quantity_produced",
        col("quantity_produced").cast("int")
    )
    .withColumn(
        "start_time",
        to_timestamp(col("start_time"), "dd-MM-yyyy HH:mm")
    )
    .withColumn(
        "end_time",
        to_timestamp(col("end_time"), "dd-MM-yyyy HH:mm")
    )
    .withColumn(
        "production_status",
        lower(trim(col("production_status")))
    )
    .filter(
        (col("quantity_produced") >= 0) &
        col("start_time").isNotNull() &
        col("end_time").isNotNull() &
        (col("end_time") >= col("start_time"))
    )
)

production_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "forgeaxis.forgeaxis_silver.silver_production_orders"
    )


# ============================================
# 6. QUALITY CHECKS
# ============================================

quality_bronze_df = spark.table(
    "forgeaxis.forgeaxis_bronze.bronze_quality_checks"
)

quality_silver_df = (
    quality_bronze_df
    .dropDuplicates(["quality_check_id"])
    .filter(col("quality_check_id").isNotNull())
    .withColumn(
        "inspection_time",
        to_timestamp(col("inspection_time"), "dd-MM-yyyy HH:mm")
    )
    .withColumn(
        "defect_count",
        col("defect_count").cast("int")
    )
    .withColumn(
        "defect_type",
        lower(trim(col("defect_type")))
    )
    .withColumn(
        "quality_status",
        lower(trim(col("quality_status")))
    )
    .filter(
        col("inspection_time").isNotNull() &
        (col("defect_count") >= 0)
    )
)

quality_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "forgeaxis.forgeaxis_silver.silver_quality_checks"
    )


# ============================================
# 7. SENSOR LOGS
# ============================================

sensor_bronze_df = spark.table(
    "forgeaxis.forgeaxis_bronze.bronze_sensor_logs"
)

sensor_silver_df = (
    sensor_bronze_df
    .dropDuplicates(["sensor_log_id"])
    .filter(col("sensor_log_id").isNotNull())
    .withColumn(
        "timestamp",
        to_timestamp(col("timestamp"), "dd-MM-yyyy HH:mm")
    )
    .withColumn(
        "temperature",
        col("temperature").cast("double")
    )
    .withColumn(
        "pressure",
        col("pressure").cast("double")
    )
    .withColumn(
        "vibration",
        col("vibration").cast("double")
    )
    .withColumn(
        "humidity",
        col("humidity").cast("double")
    )
    .filter(
        col("timestamp").isNotNull() &
        col("temperature").between(0, 150) &
        (col("pressure") >= 0) &
        (col("vibration") >= 0) &
        col("humidity").between(0, 100)
    )
)

sensor_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "forgeaxis.forgeaxis_silver.silver_sensor_logs"
    )


# ============================================
# CHECK SILVER TABLES
# ============================================

spark.sql("""
SHOW TABLES IN forgeaxis.forgeaxis_silver
""").show(truncate=False)

print("Bronze → Silver pipeline completed successfully!")